In [3]:
import xgboost as xgb
from sklearn.datasets import fetch_california_housing as fch
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from GPyOpt.methods import BayesianOptimization
from sklearn.model_selection import cross_val_score
import numpy as np

In [2]:
%pip install matplotlib

In [1]:
!pip install xgboost scikit-learn numpy==1.23.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 61.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [2]:
%pip install GPyOpt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 7.9 MB/s eta 0:00:00
  Created wheel for GPyOpt: filename=GPyOpt-1.2.6-py3-none-any.whl size=83602 sha256=fc1666d690bc85e088413dc2d33f04f2892934c2581b6a9a9dd329ed03e59c9f
  Stored in directory: /root/.cache/pip/wheels/e8/75/76/e137e5a1b3f42d86877e080683492f6c003f50ad2a73ffb534
Successfully built GPyOpt
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successful

In [4]:
# Load dataset
california_housing = fch()

X = california_housing.data
y = california_housing.target

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                	test_size=0.3,
                                                	random_state=2024)

In [5]:
print(f"Training Shape: {X_train.shape}")
print(f"Testing Shape: {X_test.shape}")


Training Shape: (14448, 8)
Testing Shape: (6192, 8)


In [6]:
param_dist = {
	'max_depth': [3, 10, 5, 15],
	'min_child_weight': [1, 5, 10],
	'subsample': [0.5, 0.7, 1.0],
	'colsample_bytree': [0.5, 0.7, 1.0],
	'n_estimators': [100, 200, 300, 400],
	'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2]
}

In [7]:
xgb_reg = xgb.XGBRegressor()

In [8]:
random_search = RandomizedSearchCV(
	xgb_reg, param_distributions=param_dist, n_iter=25,
	scoring='neg_mean_squared_error', cv=3, verbose=1, random_state=2024
)

In [9]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 25 candidates, totalling 75 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                          multi_strategy=None,
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=25,
                   param_distributions={'colsample_bytree': [0.5, 0.7, 1.0],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.15,
                                                          0.2],
                                        'max_depth': [3, 10, 5, 15],
                                        'min_child_weight': [1, 5, 10],
                                        'n_estimators': [100, 200, 300, 400],
                                        'subsample': [0.5, 0.7, 1.0]},
                   random_state=2024, scoring='neg_mean_squared_error',
                   verbose=1)

In [10]:
print("Random Search Best Parameters:", random_search.best_params_)

Random Search Best Parameters: {'subsample': 1.0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.15, 'colsample_bytree': 0.7}


In [12]:
params_random_search = {'subsample': 0.5,
                      'n_estimators': 400,
                      'min_child_weight': 5,
                      'max_depth': 10,
                      'learning_rate': 0.05,
                      'colsample_bytree': 1.0
                      }

In [13]:
# Initialize and train the model
model_random_search = xgb.XGBRegressor(**params_random_search)
model_random_search.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=1.0, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=10,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

In [14]:
predictions_random_search = model_random_search.predict(X_test)
mse_random_search = mean_squared_error(y_test, predictions_random_search)
print("MSE for Random Search: ", mse_random_search)

MSE for Random Search:  0.22042313924570814


In [15]:
baysian_opt_bounds = [
	{'name': 'max_depth', 'type': 'discrete', 'domain': (3, 10, 5, 15)},
	{'name': 'min_child_weight', 'type': 'discrete', 'domain': (1, 5, 10)},
	{'name': 'subsample', 'type': 'continuous', 'domain': (0.5, 1.0)},
	{'name': 'colsample_bytree', 'type': 'continuous', 'domain': (0.5, 1.0)},
	{'name': 'n_estimators', 'type': 'discrete', 'domain': (100, 200, 300, 400)},
	{'name': 'learning_rate', 'type': 'continuous', 'domain': (0.01, 0.2)}
]

In [16]:
def xgb_cv_score(parameters):
	parameters = parameters[0]
	score = -cross_val_score(
            	xgb.XGBRegressor(
                	max_depth=int(parameters[0]),
                	min_child_weight=int(parameters[1]),
                	subsample=parameters[2],
                	colsample_bytree=parameters[3],
                	n_estimators=int(parameters[4]),
                	learning_rate=parameters[5]),
            	X_train, y_train, scoring='neg_mean_squared_error', cv=3).mean()
	return score

In [17]:
optimizer = BayesianOptimization(
	f=xgb_cv_score, domain=baysian_opt_bounds, model_type='GP',
	acquisition_type='EI', max_iter=25
)
optimizer.run_optimization()

In [19]:
# Best parameters (as returned by GPyOpt)
best_params = optimizer.x_opt

# Map parameter names to values
param_names = [d['name'] for d in baysian_opt_bounds]
best_params_bayesian = dict(zip(param_names, best_params))

# Cast discrete parameters to int
int_params = ['max_depth', 'min_child_weight', 'n_estimators']
for p in int_params:
    if p in best_params_bayesian:
        best_params_bayesian[p] = int(best_params_bayesian[p])

print(best_params_bayesian)

{'max_depth': 10, 'min_child_weight': 5, 'subsample': 0.919534655329511, 'colsample_bytree': 0.7203721962717711, 'n_estimators': 400, 'learning_rate': 0.08716894532437484}


In [ ]:
print("Bayesian Optimization Best Parameters:", best_params_bayesian)

In [20]:
params_bayesian_opt = {
	'max_depth': 10,
	'min_child_weight': 5,
	'subsample': 0.919534655329511,
	'colsample_bytree': 0.7203721962717711,
	'n_estimators': 400,
	'learning_rate': 0.08716894532437484
}

# Initialize and train the model
model_bayesian_opt = xgb.XGBRegressor(**params_bayesian_opt)
model_bayesian_opt.fit(X_train, y_train)

# Make predictions and evaluate
predictions_bayesian_opt = model_bayesian_opt.predict(X_test)
mse_bayesian_opt = mean_squared_error(y_test, predictions_bayesian_opt)
print("MSE for Bayesian Optimization: ", mse_bayesian_opt)

MSE for Bayesian Optimization:  0.20757564122303165
